# Distillation, step 1 — cut the owner's recordings into the corpus

Roadmap §B, D101. The owner uploads `distill.zip`, many `<channel_name>_<NN>.mp3`, to the root of
the dataset repo. This notebook cuts every recording **exactly as ingest cuts an episode**: stage 1
(two-pass loudness normalisation to 16 kHz mono FLAC) and stage 2 (Silero VAD, 2–20 s slices),
run from the harness's own modules, fetched at a pinned commit and checked by sha256, not
re-implemented. Nothing touches Postgres, the queue or a paid route.

**What it writes**, to `DATASET_REPO/distill/`: each recording whole as `episodes/<id>.flac`, its
clip times in `sources/<id>.json`, and at the end `clips.jsonl` (every clip, named and timed as the
labelled export's rows) and `summary.json`. `ftkit.AudioStore` cuts clips from the whole recordings
in RAM, as it does for the labelled export, so the repo holds one file per recording rather than
one per clip. `Teacher.ipynb` reads it next.

**Gold.** The owner vouches that these channels are new (2026-09-25), so there is no voiceprint
screen here. Gold's shows are still refused by file name, which costs nothing.

**Runtime.** CPU work, but run on the A100 runtime: it has 12 cores where a CPU runtime has 2.
One process per core, each with a single-threaded VAD.

**Small blast radius.** Files are processed a chunk at a time and each chunk is uploaded as soon
as it is cut; a rerun skips every recording already in `distill/sources/`, so a lost runtime costs
at most one chunk. A file that cannot be read is reported and skipped, never fatal.

## Config

In [ ]:
DATASET_REPO = "Sagyam/nepanglish-asr"   # the owner uploads ZIP_NAME to its root
ZIP_NAME = "distill.zip"
PREFIX = "distill"                        # DATASET_REPO/distill/: episodes/, sources/, clips.jsonl
BLOCKED_CHANNELS = ["chill pill", "prime television"]   # gold's shows (D101), matched by file name
HARNESS_COMMIT = "ec60b0cb67a6d2759e0fe649fcf61fd6790769a3"     # ingest's cutting code is fetched at this commit
HARNESS_SHA256 = {
    "backend/app/services/silero_vad.py": "e09236e210a877602268c1c7176b7adfa7e22057ddb0593227fb8479cbb64c42",
    "backend/app/services/ingest/audio.py": "09eca8e1b25b7844669ad2a3ea71a3458782d74599e5cc61337fd3f316448f17",
    "backend/app/services/models/silero_vad.onnx": "1a153a22f4509e292a94e67d6f9b85e8deb25b4988682b7e174c65279d8788e3",
    "backend/app/utils/logging.py": "a4bdc2a826bc3e5b90b034d32cdbd26193484a35b2ddcbb3bb3f5cdd271f517f",
    "backend/app/utils/hashing.py": "23ab0387ab837201f89b77b05fd18140ee0df85caf0056b916af103fc55f92a4"
}
WORKERS = None                            # processes; None for every core
CHUNK = None                              # recordings per uploaded chunk; None for WORKERS

## Setup

In [ ]:
%pip install -q structlog onnxruntime
import json
import os
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
# Secrets only work from a cell run in the Colab UI: run this cell by hand once when cells are
# driven from outside (the Colab MCP); the token is then kept in the hub's token file on the VM.
if IN_COLAB:
    from google.colab import userdata

    TOKEN_FILE = Path.home() / ".cache" / "huggingface" / "token"
    if not os.environ.get("HF_TOKEN"):
        os.environ["HF_TOKEN"] = (TOKEN_FILE.read_text().strip() if TOKEN_FILE.exists()
                                  else userdata.get("HF_TOKEN"))
    if not TOKEN_FILE.exists():
        TOKEN_FILE.parent.mkdir(parents=True, exist_ok=True)
        TOKEN_FILE.write_text(os.environ["HF_TOKEN"])
        TOKEN_FILE.chmod(0o600)
FT = Path("/content/ft") if IN_COLAB else Path.cwd() / ".cache-ft"
FT.mkdir(parents=True, exist_ok=True)
print("cores:", os.cpu_count(), "| ffmpeg:", os.popen("ffmpeg -version").readline().strip())

In [ ]:
%%writefile /content/ft/ftkit.py
"""Shared fine-tuning kit for the Nepanglish ASR notebook (04c).

Everything that is not model-specific: the dataset held in RAM, duration-bucketed batches, the
harness scorer, a GPU utilisation monitor, a batch-size probe and one training loop. The notebook
writes this file out, so it imports exactly the same code as the main kernel. Keep it importable
on Python 3.10+ with numpy 1.x or 2.x.

How the GPU is kept busy:
  * audio is decoded once into RAM as int16; a clip is a slice, so no disk I/O in the loop;
  * batches are built by duration and padded to whole seconds, so there is little padding and
    only ~20 distinct shapes (cudnn.benchmark can then pick kernels once per shape);
  * DataLoader workers collate and pin the next batches while the GPU runs the current one;
  * a probe finds the largest micro-batch that survives forward+backward at the longest clip,
    with the optimizer state already allocated, and training uses a fixed fraction of it;
  * bf16 autocast, TF32 matmuls and fused AdamW; gradient accumulation reaches the effective
    batch; utilisation, throughput and padding waste are logged, not assumed.
"""

from __future__ import annotations

import json
import math
import random
import re
import shutil
import subprocess
import sys
import threading
import time
from collections import Counter
from collections.abc import Callable, Iterator, Sequence
from contextlib import contextmanager
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Any

import numpy as np
import soundfile as sf
import torch

REPO = "Sagyam/nepanglish-asr"
SR = 16_000


# --- data --------------------------------------------------------------------------------------


def download_dataset(local: str | None = None) -> Path:
    """The HF dataset snapshot, or a local copy in the same layout."""
    if local:
        return Path(local)
    from huggingface_hub import snapshot_download

    return Path(
        snapshot_download(
            REPO,
            repo_type="dataset",
            allow_patterns=["training/*", "gold/*", "harness/*", "analytics/*"],
        )
    )


def load_splits(data: Path) -> dict[str, list[dict]]:
    """train / val from the training export, gold from the gold export; asserts disjointness."""
    rows = [
        json.loads(line) for line in (data / "training" / "training.jsonl").open(encoding="utf-8")
    ]
    gold = [json.loads(line) for line in (data / "gold" / "gold.jsonl").open(encoding="utf-8")]
    splits = {
        "train": [r for r in rows if r["split"] == "train"],
        "val": [r for r in rows if r["split"] == "val"],
        "gold": gold,
    }
    trained = {r["segment_id"] for r in rows}
    assert not trained & {r["segment_id"] for r in gold}, "a gold clip is in the training export"
    return splits


def attach_speaker_turns(data: Path, rows: Sequence[dict]) -> int:
    """Copy each row's diarized turns, with their linked voices (D78, D87), from the analytics
    export: the training export does not carry them. Returns how many rows have turns."""
    want = {r["segment_id"]: r for r in rows}
    with (data / "analytics" / "analytics.jsonl").open(encoding="utf-8") as fh:
        for line in fh:
            a = json.loads(line)
            if a["segment_id"] in want:
                want[a["segment_id"]]["speaker_turns"] = a.get("speaker_turns")
    return sum(bool(r.get("speaker_turns")) for r in rows)


def duration(row: dict) -> float:
    return row["end_time"] - row["start_time"]


class AudioStore:
    """Every episode decoded once into RAM as int16; a clip is a slice of it (a view, no copy).

    ``folder`` is where the whole recordings are, under ``data``: the labelled export's
    ``training/episodes`` by default, or the distillation corpus's ``distill/episodes`` (D101)."""

    def __init__(self, data: Path, episode_ids: Sequence[str], folder: str = "training/episodes"):
        self.audio: dict[str, np.ndarray] = {}
        for ep in sorted(set(episode_ids)):
            audio, sr = sf.read(data / folder / f"{ep}.flac", dtype="int16")
            assert sr == SR and audio.ndim == 1, (ep, sr, audio.shape)
            self.audio[ep] = audio

    def clip(self, row: dict) -> np.ndarray:
        audio = self.audio[row["episode_id"]]
        return audio[round(row["start_time"] * SR) : round(row["end_time"] * SR)]

    def clip_f32(self, row: dict) -> torch.Tensor:
        return torch.from_numpy(self.clip(row).astype(np.float32) / 32768.0)

    @property
    def gib(self) -> float:
        return sum(a.nbytes for a in self.audio.values()) / 2**30


def bucket_batches(
    rows: Sequence[dict],
    *,
    budget_s: float,
    max_items: int,
    pad_to_s: float,
    shuffle: bool,
    seed: int = 0,
) -> list[list[int]]:
    """Index batches whose padded size (items x longest clip, rounded up to `pad_to_s`) fits
    `budget_s`. Sorting by duration keeps padding low; shuffling reorders whole batches and
    jitters lengths slightly so the same clips do not always share a batch."""
    rng = random.Random(seed)
    jitter = [rng.uniform(-0.3, 0.3) if shuffle else 0.0 for _ in rows]
    order = sorted(range(len(rows)), key=lambda i: duration(rows[i]) + jitter[i])
    batches: list[list[int]] = []
    cur: list[int] = []
    longest = 0.0
    for i in order:
        d = math.ceil(duration(rows[i]) / pad_to_s) * pad_to_s
        if cur and ((len(cur) + 1) * max(longest, d) > budget_s or len(cur) >= max_items):
            batches.append(cur)
            cur, longest = [], 0.0
        cur.append(i)
        longest = max(longest, d)
    if cur:
        batches.append(cur)
    if shuffle:
        rng.shuffle(batches)
    return batches


def pad_len(samples: int, pad_to_s: float) -> int:
    step = int(pad_to_s * SR)
    return math.ceil(samples / step) * step


class _Batches(torch.utils.data.Dataset):
    def __init__(self, rows, batches, collate):
        self.rows, self.batches, self.collate = rows, batches, collate

    def __len__(self):
        return len(self.batches)

    def __getitem__(self, i):
        return self.collate([self.rows[j] for j in self.batches[i]])


def loader(rows, batches, collate, workers: int) -> torch.utils.data.DataLoader:
    return torch.utils.data.DataLoader(
        _Batches(rows, batches, collate),
        batch_size=None,
        shuffle=False,
        num_workers=workers,
        pin_memory=True,
        prefetch_factor=4 if workers else None,
        persistent_workers=False,
    )


# --- scoring -----------------------------------------------------------------------------------


def is_loop(text: str) -> bool:
    """A 3-word sequence repeated 5 or more times: the decoder is stuck, not transcribing."""
    toks = text.split()
    top = Counter(zip(toks, toks[1:], toks[2:], strict=False)).most_common(1)
    return bool(top) and top[0][1] >= 5


class RetryLoops:
    """Wraps a batch `transcribe`: a clip whose first decode loops is decoded again, alone, by
    `retry` (anti-repetition settings); every other clip keeps its first decode untouched.
    `log` keeps (segment_id, first, retried), so the effect is measurable within one run."""

    def __init__(self, transcribe: Callable[[list[dict]], list[str]], retry: Callable[[dict], str]):
        self.transcribe, self.retry = transcribe, retry
        self.log: list[tuple[str, str, str]] = []

    def __call__(self, rows: list[dict]) -> list[str]:
        texts = self.transcribe(rows)
        for i, text in enumerate(texts):
            if is_loop(text):
                texts[i] = self.retry(rows[i])
                self.log.append((rows[i]["segment_id"], text, texts[i]))
        return texts


def harness_scorer(data: Path, work: Path) -> Callable[[Sequence[str], Sequence[str]], dict]:
    """fold.py from the dataset's harness/, imported from a real copy: the HF cache stores files as
    symlinks into its blob store, and normalize.py finds its config via its resolved path.

    Two layouts are accepted: the repo's (backend/app/services/, config/), and the flat one the
    2026-09-21 export uploaded (fold.py, normalize.py, normalization.yaml side by side), which
    is laid back out here because normalize.py looks for ../../../config/."""
    src, dst = data / "harness", work / "harness"
    if not dst.exists():
        if (src / "fold.py").exists():
            (dst / "backend" / "app" / "services").mkdir(parents=True)
            (dst / "config").mkdir()
            for name in ("fold.py", "normalize.py"):
                shutil.copyfile(src / name, dst / "backend" / "app" / "services" / name)
            shutil.copyfile(src / "normalization.yaml", dst / "config" / "normalization.yaml")
        else:
            shutil.copytree(src, dst)
    for name in [m for m in sys.modules if m == "app" or m.startswith("app.")]:
        del sys.modules[name]
    sys.path.insert(0, str(dst / "backend"))
    from app.services.fold import fold_tokens, word_errors
    from rapidfuzz.distance import Levenshtein

    dev = re.compile(r"[ऀ-ॿ]")

    def chars(text: str) -> str:
        return " ".join(t if dev.search(t) else t.lower() for t in fold_tokens(text))

    def per_clip(refs: Sequence[str], hyps: Sequence[str]) -> list[dict]:
        """Every count a score needs, per clip, from one alignment each: `summarize` adds any
        subset of them up, so scoring per class or pairing against another system aligns
        nothing again. errors = sub + del + ins (folded)."""
        out = []
        for ref, hyp in zip(refs, hyps, strict=True):
            folded = word_errors(ref, hyp)
            raw = word_errors(ref, hyp, folded=False)
            ref_chars, hyp_chars = chars(ref), chars(hyp)
            out.append(
                {
                    "words": len(fold_tokens(ref)),
                    "errors": folded.errors,
                    "sub": folded.substitutions,
                    "del": folded.deletions,
                    "ins": folded.insertions,
                    "raw_words": raw.ref_words,
                    "raw_errors": raw.errors,
                    "chars": len(ref_chars),
                    "char_errors": Levenshtein.distance(ref_chars, hyp_chars),
                    "loop": int(is_loop(hyp)),
                }
            )
        return out

    def summarize(clips: Sequence[dict]) -> dict:
        """The score of a set of clips from their `per_clip` counts."""
        t = {k: sum(c[k] for c in clips) for k in clips[0]} if clips else Counter()
        words = max(t["words"], 1)
        folded = t["sub"] + t["del"] + t["ins"]
        assert folded == t["errors"], "S + D + I must add up to the folded errors"
        return {
            "wer": 100 * t["errors"] / words,
            "raw_wer": 100 * t["raw_errors"] / max(t["raw_words"], 1),
            "cer": 100 * t["char_errors"] / max(t["chars"], 1),
            # folded, per 100 reference words. In crosstalk a label keeps what was audible, which is
            # not always both voices, so a model that writes the other one can be charged insertions
            "sub": 100 * t["sub"] / words,
            "del": 100 * t["del"] / words,
            "ins": 100 * t["ins"] / words,
            "loops": t["loop"],
            "clips": len(clips),
        }

    def score(refs: Sequence[str], hyps: Sequence[str]) -> dict:
        return summarize(per_clip(refs, hyps))

    score.per_clip = per_clip
    score.summarize = summarize
    return score


def transcribe_rows(
    rows: Sequence[dict],
    transcribe: Callable[[list[dict]], list[str]],
    *,
    budget_s: float,
    max_items: int,
    pad_to_s: float,
) -> tuple[list[str], list[float]]:
    """Run `transcribe` over duration-bucketed batches; returns texts and per-clip compute seconds
    in the original row order."""
    texts: list[str] = [""] * len(rows)
    compute = [0.0] * len(rows)
    for batch in bucket_batches(
        rows, budget_s=budget_s, max_items=max_items, pad_to_s=pad_to_s, shuffle=False
    ):
        torch.cuda.synchronize()
        t0 = time.perf_counter()
        out = transcribe([rows[i] for i in batch])
        torch.cuda.synchronize()
        took = time.perf_counter() - t0
        audio_s = sum(duration(rows[i]) for i in batch)
        for i, text in zip(batch, out, strict=True):
            texts[i] = text.strip()
            compute[i] = took * duration(rows[i]) / audio_s
    return texts, compute


def write_hyps(
    path: Path, rows: Sequence[dict], texts: Sequence[str], compute: Sequence[float]
) -> None:
    """The hypothesis-cache format the bake-off comparisons used, so the result can join them."""
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as fh:
        for r, text, c in zip(rows, texts, compute, strict=True):
            fh.write(
                json.dumps(
                    {"segment_id": r["segment_id"], "text": text, "compute_s": c},
                    ensure_ascii=False,
                )
                + "\n"
            )


# --- GPU ---------------------------------------------------------------------------------------


def fast_cuda() -> None:
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    torch.backends.cudnn.benchmark = True  # safe: shapes are padded to whole seconds
    torch.set_float32_matmul_precision("high")


class GpuMonitor:
    """Samples nvidia-smi in a background thread; `window()` returns the mean utilisation and the
    peak memory since the previous call."""

    def __init__(self, every: float = 1.0):
        self.every, self._util, self._mem = every, [], []
        self._stop = threading.Event()
        self._thread = threading.Thread(target=self._run, daemon=True)

    def _run(self):
        while not self._stop.is_set():
            try:
                out = subprocess.run(
                    [
                        "nvidia-smi",
                        "--query-gpu=utilization.gpu,memory.used",
                        "--format=csv,noheader,nounits",
                    ],
                    capture_output=True,
                    text=True,
                    timeout=5,
                )
                util, mem = (float(x) for x in out.stdout.strip().splitlines()[0].split(","))
                self._util.append(util)
                self._mem.append(mem)
            except Exception:
                pass
            self._stop.wait(self.every)

    def start(self) -> GpuMonitor:
        self._thread.start()
        return self

    def stop(self) -> None:
        self._stop.set()

    def window(self) -> dict:
        util, mem = self._util, self._mem
        self._util, self._mem = [], []
        return {
            "gpu_util": float(np.mean(util)) if util else float("nan"),
            "gpu_mem_gib": max(mem) / 1024 if mem else float("nan"),
        }


def init_optimizer_state(model: torch.nn.Module, optimizer: torch.optim.Optimizer) -> None:
    """Allocate AdamW's moment buffers before probing, without moving any weight: with zero
    gradients and zero weight decay an AdamW step is a no-op, but it creates the state."""
    decay = [g["weight_decay"] for g in optimizer.param_groups]
    for g in optimizer.param_groups:
        g["weight_decay"] = 0.0
    for p in model.parameters():
        if p.requires_grad:
            p.grad = torch.zeros_like(p)
    optimizer.step()
    optimizer.zero_grad(set_to_none=True)
    for g, d in zip(optimizer.param_groups, decay, strict=True):
        g["weight_decay"] = d
    for state in optimizer.state.values():
        if "step" in state:
            state["step"].zero_()


def probe_max_items(
    step: Callable[[int], None], lo: int, hi: int, params: Sequence[torch.nn.Parameter]
) -> int:
    """The largest n in [lo, hi] for which `step(n)` (one forward+backward at the worst case)
    fits in memory. `step` must not touch the gradients.

    The gradient buffer stays allocated throughout, as it does in training from the second
    micro-batch of every accumulated step: freeing it between probes measured the activations
    without it and picked a batch that ran out of memory once training accumulated."""
    for p in params:
        p.grad = torch.zeros_like(p)
    best = 0
    while lo <= hi:
        mid = (lo + hi) // 2
        try:
            step(mid)
            torch.cuda.synchronize()
            best, lo = mid, mid + 1
        except torch.cuda.OutOfMemoryError:
            hi = mid - 1
        for p in params:
            p.grad.zero_()
        torch.cuda.empty_cache()
    for p in params:
        p.grad = None
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    return best


# --- training ----------------------------------------------------------------------------------


@dataclass
class TrainConfig:
    name: str
    out: str
    epochs: int = 10
    lr: float = 1e-5
    schedule: str = "linear"  # linear warmup+decay, or "tristage" (fairseq2's default)
    warmup_frac: float = 0.1
    effective_s: float = 720.0  # audio seconds per optimizer step, reached by accumulation
    weight_decay: float = 0.0
    clip_norm: float = 1.0
    patience: int = 3  # evaluations without a val-WER improvement before stopping
    # WER points an evaluation must gain on the last counted one to reset `patience`. 0 counts
    # any gain, so a crawl of 0.2 points an epoch never stops. The best weights are kept either way.
    min_delta: float = 0.0
    seed: int = 0
    log_every: int = 10
    workers: int = 6


def lr_factor(step: int, total: int, cfg: TrainConfig) -> float:
    warm = max(1, int(cfg.warmup_frac * total))
    if step < warm:
        return (step + 1) / warm
    if cfg.schedule == "tristage":  # 10% warmup, 40% hold, 50% exponential decay to 5%
        hold_end = int(0.5 * total)
        if step < hold_end:
            return 1.0
        frac = (step - hold_end) / max(1, total - hold_end)
        return math.exp(math.log(0.05) * frac)
    return max(0.0, (total - step) / max(1, total - warm))


def group_steps(rows, batches, effective_s: float) -> list[list[list[int]]]:
    """Micro-batches grouped into optimizer steps of about `effective_s` seconds of real audio."""
    steps, cur, acc = [], [], 0.0
    for b in batches:
        cur.append(b)
        acc += sum(duration(rows[i]) for i in b)
        if acc >= effective_s:
            steps.append(cur)
            cur, acc = [], 0.0
    if cur:
        steps.append(cur)
    return steps


@contextmanager
def timed(label: str) -> Iterator[None]:
    """Print `label...` before a step and `label: done in N s` after it, so a step that stalls
    shows which one it is instead of leaving a cell silent."""
    print(f"{label}...", flush=True)
    t0 = time.perf_counter()
    yield
    print(f"{label}: done in {time.perf_counter() - t0:.0f} s", flush=True)


def sid(m: dict) -> str:
    """A score's error split, per 100 reference words: S + D + I is its folded WER. Read all three
    on crosstalk: the labels drop the other voice, so hearing it costs insertions."""
    return f"S {m['sub']:.2f}  D {m['del']:.2f}  I {m['ins']:.2f}"


def retry_note(val: dict) -> str:
    """For an `evaluate` that wraps its decode in RetryLoops and reports the greedy score too."""
    if "retried" not in val:
        return ""
    return f"  (greedy WER {val['greedy_wer']:.2f}, {val['retried']} retried)"


def speed_check(
    model: torch.nn.Module,
    *,
    rows: Sequence[dict],
    batches: list[list[int]],
    collate: Callable[[list[dict]], Any],
    loss_fn: Callable[[torch.nn.Module, Any], tuple[torch.Tensor, int]],
    evaluate: Callable[[torch.nn.Module], dict],
    val_rows: Sequence[dict],
    gold_rows: Sequence[dict],
    epochs: int,
    monitor: GpuMonitor,
    workers: int = 6,
    n_micro: int = 10,
) -> dict:
    """Time the run before committing to it: forward+backward on `n_micro` real training
    micro-batches (no optimizer step, so no weight moves), then one full val pass, which is also
    the val WER before training. Projects the wall time of every epoch and the gold pass.

    The micro-batches run twice and only the second pass is timed, so cudnn's per-shape kernel
    search and worker start-up are excluded. Gradients stay allocated between micro-batches, as
    they do under accumulation, so the memory reading is training's. BatchNorm running
    statistics are restored after."""
    bns = [m for m in model.modules() if isinstance(m, torch.nn.modules.batchnorm._BatchNorm)]
    saved = [{k: v.clone() for k, v in m.state_dict().items()} for m in bns]
    sample = batches[:n_micro]
    it = iter(loader(rows, sample + sample, collate, workers))

    def run() -> tuple[float, float, int]:
        audio = padded = 0.0
        clips = 0
        for _ in sample:
            b = next(it)
            with torch.autocast("cuda", dtype=torch.bfloat16):
                loss, _ = loss_fn(model, b)
            loss.backward()
            model.zero_grad(set_to_none=False)
            audio, padded = audio + b["seconds"], padded + b["padded_seconds"]
            clips += len(b["lens"]) if "lens" in b else b["wav"].shape[0]
        torch.cuda.synchronize()
        return audio, padded, clips

    model.train()
    run()
    monitor.window()
    t0 = time.perf_counter()
    audio, padded, clips = run()
    train_dt = time.perf_counter() - t0
    gpu = monitor.window()
    model.zero_grad(set_to_none=True)
    for m, s in zip(bns, saved, strict=True):
        m.load_state_dict(s)

    model.eval()
    torch.cuda.synchronize()
    t0 = time.perf_counter()
    with torch.no_grad():
        val = evaluate(model)
    torch.cuda.synchronize()
    val_dt = time.perf_counter() - t0
    model.train()

    secs = lambda rs: sum(duration(r) for r in rs)  # noqa: E731
    epoch_s = secs(rows) / (audio / train_dt)
    gold_s = val_dt * secs(gold_rows) / secs(val_rows)
    rec = {
        "train_x_realtime": audio / train_dt,
        "train_ms_per_clip": 1000 * train_dt / clips,
        "padding_waste": 1 - audio / padded,
        **gpu,
        "val_s": val_dt,
        "val_ms_per_clip": 1000 * val_dt / len(val_rows),
        **{f"val_before_{k}": v for k, v in val.items()},
        "epoch_min": epoch_s / 60,
        "projected_h": (epochs * (epoch_s + val_dt) + gold_s) / 3600,
    }
    print(
        f"train: {rec['train_x_realtime']:.0f}x realtime = "
        f"{rec['train_ms_per_clip']:.0f} ms per clip "
        f"(forward+backward, {clips} clips), pad waste {rec['padding_waste']:.0%}, "
        f"GPU {rec['gpu_util']:.0f}% util, {rec['gpu_mem_gib']:.1f} GiB\n"
        f"val:   {val_dt:.0f} s for {len(val_rows)} clips = "
        f"{rec['val_ms_per_clip']:.0f} ms per clip "
        f"(batched decode); WER before training {val['wer']:.2f} ({sid(val)}), loops {val['loops']}"
        f"{retry_note(val)}\n"
        f"projected: {rec['epoch_min']:.1f} min per epoch + {val_dt / 60:.1f} min val -> "
        f"at most {rec['projected_h']:.2f} h for {epochs} epochs and gold "
        "(early stopping can cut it)",
        flush=True,
    )
    return rec


def train(
    model: torch.nn.Module,
    *,
    cfg: TrainConfig,
    rows: Sequence[dict],
    make_batches: Callable[[int], list[list[int]]],
    collate: Callable[[list[dict]], Any],
    loss_fn: Callable[[torch.nn.Module, Any], tuple[torch.Tensor, int]],
    evaluate: Callable[[torch.nn.Module], dict],
    save_best: Callable[[torch.nn.Module], None],
    optimizer: torch.optim.Optimizer,
    monitor: GpuMonitor,
) -> dict:
    """Train with bf16 autocast and gradient accumulation; evaluate on val after every epoch;
    keep the best weights (by val WER) in CPU memory and hand them to `save_best`. A hand stop
    (KeyboardInterrupt) after the first evaluation ends training the same way.

    `loss_fn` returns a *summed* loss and the number of units it sums over (clips for CTC, tokens
    for cross-entropy); gradients are divided by the step's total units, so accumulated
    micro-batches of different sizes are weighted exactly. Count the units on the CPU (in
    `collate`): an `int()` of a GPU tensor stalls the host once per micro-batch."""
    out = Path(cfg.out)
    out.mkdir(parents=True, exist_ok=True)
    torch.manual_seed(cfg.seed)
    plan = [group_steps(rows, make_batches(e), cfg.effective_s) for e in range(cfg.epochs)]
    total = sum(len(p) for p in plan)
    base_lrs = [g["lr"] for g in optimizer.param_groups]
    params = [p for p in model.parameters() if p.requires_grad]
    history, best, best_state, bad, step = [], float("inf"), None, 0, 0
    counted = float("inf")  # the val WER that last reset `bad`
    print(
        f"{cfg.name}: {total} optimizer steps over {cfg.epochs} epochs "
        f"(~{cfg.effective_s / 60:.0f} min of audio each), peak lr {cfg.lr:g}, {cfg.schedule}"
    )
    stopped_by_hand = False
    try:
        for epoch in range(cfg.epochs):
            model.train()
            flat = [b for s in plan[epoch] for b in s]
            sizes = [len(s) for s in plan[epoch]]
            it = iter(loader(rows, flat, collate, cfg.workers))
            t_log, audio_log, padded_log, loss_log, units_log = (
                time.perf_counter(),
                0.0,
                0.0,
                0.0,
                0,
            )
            monitor.window()
            for n_micro in sizes:
                for g, lr0 in zip(optimizer.param_groups, base_lrs, strict=True):
                    g["lr"] = lr0 * lr_factor(step, total, cfg)
                step_units = 0
                for _ in range(n_micro):
                    batch = next(it)
                    with torch.autocast("cuda", dtype=torch.bfloat16):
                        loss, units = loss_fn(model, batch)
                    loss.backward()
                    step_units += units
                    loss_log += loss.detach()  # stays on the GPU: no host sync per micro-batch
                    units_log += units
                    audio_log += batch["seconds"]
                    padded_log += batch["padded_seconds"]
                torch._foreach_div_(
                    [p.grad for p in params if p.grad is not None], max(step_units, 1)
                )
                gnorm = torch.nn.utils.clip_grad_norm_(params, cfg.clip_norm)
                optimizer.step()
                optimizer.zero_grad(set_to_none=True)
                step += 1
                if step % cfg.log_every == 0:
                    torch.cuda.synchronize()
                    dt = time.perf_counter() - t_log
                    gpu = monitor.window()
                    rec = {
                        "step": step,
                        "epoch": epoch + 1,
                        "lr": optimizer.param_groups[0]["lr"],
                        "loss": float(loss_log) / max(units_log, 1),
                        "grad_norm": float(gnorm),
                        "audio_x_realtime": audio_log / dt,
                        "padding_waste": 1 - audio_log / max(padded_log, 1e-9),
                        "peak_alloc_gib": torch.cuda.max_memory_allocated() / 2**30,
                        **gpu,
                    }
                    history.append(rec)
                    print(
                        f"step {step:5d}/{total} ep {epoch + 1} lr {rec['lr']:.2e} "
                        f"loss {rec['loss']:.4f} | {rec['audio_x_realtime']:6.0f}x realtime, "
                        f"pad waste {rec['padding_waste']:.0%}, "
                        f"GPU {rec['gpu_util']:.0f}% util, {rec['gpu_mem_gib']:.1f} GiB",
                        flush=True,
                    )
                    t_log, audio_log, padded_log, loss_log, units_log = (
                        time.perf_counter(),
                        0.0,
                        0.0,
                        0.0,
                        0,
                    )
            model.eval()
            with torch.no_grad():
                val = evaluate(model)
            history.append(
                {"epoch": epoch + 1, "step": step, **{f"val_{k}": v for k, v in val.items()}}
            )
            improved = val["wer"] < best
            print(
                f"== epoch {epoch + 1}: val WER {val['wer']:.2f} ({sid(val)})  "
                f"CER {val['cer']:.2f}  raw WER {val['raw_wer']:.2f}  "
                f"loops {val['loops']}{retry_note(val)}" + ("  (best)" if improved else ""),
                flush=True,
            )
            (out / "history.json").write_text(json.dumps(history, indent=1))
            if improved:
                best_state = {
                    k: v.detach().to("cpu", copy=True) for k, v in model.state_dict().items()
                }
                best = val["wer"]
            if val["wer"] < counted - cfg.min_delta:  # at 0: the old rule, any strict gain
                counted, bad = val["wer"], 0
            else:
                bad += 1
                if bad >= cfg.patience:
                    print(
                        f"val WER gained less than {cfg.min_delta:g} points in {cfg.patience} "
                        "evaluations; stopping"
                    )
                    break
    except KeyboardInterrupt:
        # Colab's stop button: end training here and keep the best weights so far, so they are
        # saved and scored like a finished run's. Before any evaluation there are none to keep.
        if best_state is None:
            raise
        stopped_by_hand = True
        print(f"stopped by hand; keeping the best weights (val WER {best:.2f})", flush=True)
    monitor.window()
    with timed("saving the best weights"):
        if best_state is not None:
            model.load_state_dict(best_state)
        model.eval()
        save_best(model)
    (out / "config.json").write_text(json.dumps(asdict(cfg), indent=1))
    return {
        "best_val_wer": best,
        "steps": step,
        "history": history,
        "stopped_by_hand": stopped_by_hand,
    }

In [ ]:
%%writefile /content/ft/distill.py
"""Pure helpers for the distillation notebook (roadmap §B).

No torch and no NeMo, so the rules that decide what shapes the students' tokenizer and what their
scores are paired against are tested from backend/tests. Keep it importable on Python 3.10+.
"""

from __future__ import annotations

import re
import unicodedata
from collections.abc import Callable, Mapping, Sequence
from typing import Any


def tokenizer_texts(splits: Mapping[str, Sequence[dict]]) -> list[str]:
    """The labels the shared student tokenizer is trained on: train only.

    Val and gold text never shapes the vocabulary, so a word only they contain is scored as the
    student would meet it on new audio. Raises if a val or gold clip is in train."""
    held = {r["segment_id"] for name in ("val", "gold") for r in splits.get(name, ())}
    leaked = sorted(held & {r["segment_id"] for r in splits["train"]})
    if leaked:
        raise ValueError(f"held-out clips in train: {leaked[:5]}")
    return [r["text"] for r in splits["train"]]


def reference_texts(rows: Sequence[dict], by_id: Mapping[str, str]) -> list[str]:
    """Another system's transcripts in `rows`' order, for pairing clip by clip. Raises, naming
    them, if any row has none: a pairing over a subset would not be the same clips."""
    missing = [r["segment_id"] for r in rows if r["segment_id"] not in by_id]
    if missing:
        raise ValueError(f"{len(missing)} clip(s) have no reference transcript: {missing[:5]}")
    return [by_id[r["segment_id"]] for r in rows]


def by_class(
    rows: Sequence[dict],
    clips: Sequence[Any],
    summarize: Callable[[list[Any]], Any],
    keys: Sequence[str] | None = None,
) -> dict[str, dict[str, Any]]:
    """`summarize` per value of each clip class (D87), as {key: {value: summary}}.

    `clips` holds one per-clip count per row (`score.per_clip`), aligned once by the caller: a
    group's score is its clips' counts added up, so no clip is aligned again per key. `keys`
    defaults to every class key any row carries. A clip without a key (or with no classes) is left
    out of that key's groups, not counted under a made-up value."""
    if len(rows) != len(clips):
        raise ValueError("rows and clips must describe the same clips")
    if keys is None:
        keys = sorted({k for r in rows for k in (r.get("classes") or {})})
    out: dict[str, dict[str, Any]] = {}
    for key in keys:
        groups: dict[str, list[int]] = {}
        for i, r in enumerate(rows):
            value = (r.get("classes") or {}).get(key)
            if value is not None:
                groups.setdefault(str(value), []).append(i)
        if groups:
            out[key] = {v: summarize([clips[i] for i in idx]) for v, idx in sorted(groups.items())}
    return out


def filter_pseudo(
    rows: Sequence[dict],
    *,
    tokens_per_s: tuple[float, float],
    drop_fraction: float,
) -> tuple[list[dict], dict[str, Any]]:
    """Roadmap §B step 3: keep the teacher's labels worth training on, cheapest rule first.

    Each row carries ``segment_id``, ``text``, ``n_tokens`` (teacher tokens), ``duration`` (s),
    ``mean_logprob`` and ``looped`` (the greedy output loops, ``ftkit.is_loop``). Dropped, in
    order: a clip whose output loops; an empty transcript; a rate outside ``tokens_per_s`` (the
    range train's labels span, in the teacher's tokens); then the least confident
    ``drop_fraction`` of what is left, by mean log-prob. Returns the kept rows, in input order,
    and a report of what each rule dropped.
    """
    if not 0.0 <= drop_fraction < 1.0:
        raise ValueError(f"drop_fraction must be in [0, 1), not {drop_fraction}")
    lo, hi = tokens_per_s
    dropped = {"loop": 0, "empty": 0, "rate": 0, "confidence": 0}
    left = []
    for r in rows:
        if r["looped"]:
            dropped["loop"] += 1
        elif not r["text"].strip():
            dropped["empty"] += 1
        elif not lo <= r["n_tokens"] / r["duration"] <= hi:
            dropped["rate"] += 1
        else:
            left.append(r)
    cut = None
    n_drop = int(len(left) * drop_fraction)
    if n_drop:
        least = sorted(left, key=lambda r: r["mean_logprob"])[:n_drop]
        cut = max(r["mean_logprob"] for r in least)
        gone = {r["segment_id"] for r in least}
        left = [r for r in left if r["segment_id"] not in gone]
        dropped["confidence"] = n_drop
    report = {
        "total": len(rows),
        "kept": len(left),
        "kept_hours": round(sum(r["duration"] for r in left) / 3600, 3),
        "dropped": dropped,
        "logprob_cut": cut,
        "tokens_per_s": [lo, hi],
        "drop_fraction": drop_fraction,
    }
    return left, report


def latin_share(text: str) -> float:
    """The share of a transcript's words written in Latin script: a code-mixing proxy that needs
    only the text, for pseudo-labels, which have no CMI class."""
    words = [w for w in text.split() if any(c.isalpha() for c in w)]
    if not words:
        return 0.0
    return sum(any("a" <= c.lower() <= "z" for c in w) for w in words) / len(words)


def mixing_bucket(share: float) -> str:
    """``latin_share`` in the corpus's CMI buckets (D87): 0, <15, 15-30, 30+ percent."""
    if share == 0:
        return "0"
    if share < 0.15:
        return "<15"
    return "15-30" if share < 0.30 else "30+"


# --- PreDistill: the owner's zip of recordings (D101) ------------------------------------------

_NUMBER = re.compile(r"_\d+$")


def _path_safe(text: str) -> str:
    """Letters and digits of any script, combining marks (Devanagari's vowel signs, which regex's
    word class misses), ``.``, ``-`` and ``_`` kept; every other run of characters becomes one
    ``_``."""
    kept = "".join(
        c if c.isalnum() or c in "._-" or unicodedata.category(c).startswith("M") else "\0"
        for c in text
    )
    return re.sub("\0+", "_", kept).strip("_")


def source_from_filename(name: str) -> tuple[str, str]:
    """A recording named ``<channel_name>_<NN>.mp3`` as ``(source_id, channel)``.

    The source id is the file's stem with every run of characters unsafe in a path turned into
    one ``_`` (letters of any script, digits, ``.``, ``-`` and ``_`` are kept); the channel is the
    id less its trailing ``_<number>``.
    """
    stem = name.replace("\\", "/").rsplit("/", 1)[-1].rsplit(".", 1)[0]
    source_id = _path_safe(stem)
    return source_id, _NUMBER.sub("", source_id) or source_id


def channel_blocked(channel: str, blocked: Sequence[str]) -> bool:
    """Whether a channel's name contains a blocked name, ignoring case and ``_`` against spaces."""
    name = channel.replace("_", " ").casefold()
    return any(b.strip().casefold() in name for b in blocked if b.strip())


def slice_rows(source_id: str, channel: str, slices: Sequence[tuple[float, float]]) -> list[dict]:
    """The manifest rows of one recording's VAD slices, named and timed as the labelled export's
    rows are, so ``ftkit.AudioStore`` cuts them from the whole recording."""
    return [
        {
            "segment_id": f"{source_id}_{i:05d}",
            "episode_id": source_id,
            "source_id": source_id,
            "channel": channel,
            "start_time": round(start, 3),
            "end_time": round(end, 3),
            "duration": round(end - start, 3),
        }
        for i, (start, end) in enumerate(slices)
    ]

In [ ]:
%%writefile /content/ft/prekit.py
"""PreDistill's per-file worker (roadmap §B step 1, D101).

One recording from the owner's zip goes through ingest's own stage 1 (``normalize_audio``: two-pass
loudness normalisation to 16 kHz mono FLAC) and stage 2 (Silero VAD, ``segment_audio_to_slices``:
2-20 s), imported from the harness's modules, which the notebook fetches at a pinned commit. What
it keeps is the whole normalised recording and its clip rows, the layout ``ftkit.AudioStore``
reads. One process per core: the VAD runs single-threaded here, so twelve workers do not fight
over twelve cores; its output does not depend on the thread count.
"""

from __future__ import annotations

import json
import time
from pathlib import Path
from typing import Any

import distill
import soundfile as sf

_VAD: dict[str | None, Any] = {}


def _vad(model_path: str | None):
    """This process's VAD, loaded once: the harness's SileroVAD on one thread."""
    if model_path not in _VAD:
        from app.services.silero_vad import SileroVAD

        class OneThreadVAD(SileroVAD):
            def _load_model(self) -> None:
                if not self.model_path.is_file():
                    self._session = None  # detect_turns falls back to energy, as in ingest
                    return
                import onnxruntime as ort

                opts = ort.SessionOptions()
                opts.intra_op_num_threads = 1
                opts.inter_op_num_threads = 1
                opts.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL
                self._session = ort.InferenceSession(
                    str(self.model_path), sess_options=opts, providers=["CPUExecutionProvider"]
                )

        _VAD[model_path] = OneThreadVAD(model_path)
    return _VAD[model_path]


def process_file(
    audio_path: str, out_dir: str, blocked: list[str], model_path: str | None = None
) -> dict[str, Any]:
    """Normalise and cut one recording into ``out_dir/episodes/<id>.flac`` and
    ``out_dir/sources/<id>.json`` (its metadata and clip rows).

    Returns ``status`` ``prepared``, ``blocked`` (a channel in ``distill.blocked_channels``) or
    ``failed`` (with ``error``; nothing is left behind), so one bad file never stops the run.

    Args:
        audio_path: The MP3 (or any audio ffmpeg reads), named ``<channel_name>_<NN>``.
        out_dir: Where the chunk's files go before they are uploaded.
        blocked: Channel names that may not enter the corpus.
        model_path: The Silero ONNX; None for the harness's default location.
    """
    started = time.perf_counter()
    source_id, channel = distill.source_from_filename(audio_path)
    base = {"source_id": source_id, "channel": channel, "file": Path(audio_path).name}
    if distill.channel_blocked(channel, blocked):
        return {**base, "status": "blocked"}
    from app.services.ingest.audio import normalize_audio
    from app.services.silero_vad import segment_audio_to_slices

    out = Path(out_dir)
    flac = out / "episodes" / f"{source_id}.flac"
    meta_path = out / "sources" / f"{source_id}.json"
    try:
        flac.parent.mkdir(parents=True, exist_ok=True)
        meta_path.parent.mkdir(parents=True, exist_ok=True)
        duration = normalize_audio(Path(audio_path), flac)
        audio, sample_rate = sf.read(str(flac), dtype="float32")
        vad = _vad(model_path)
        turns = vad.detect_turns(audio, sample_rate=sample_rate)
        slices = segment_audio_to_slices(turns, duration, audio=audio, sample_rate=sample_rate)
        rows = distill.slice_rows(source_id, channel, slices)
        meta = {
            **base,
            "duration": round(duration, 3),
            "clips": len(rows),
            "speech_seconds": round(sum(r["duration"] for r in rows), 3),
            "vad": "silero" if vad.available else "energy",
        }
        meta_path.write_text(json.dumps({**meta, "rows": rows}, ensure_ascii=False), "utf-8")
        return {
            **meta,
            "status": "prepared",
            "seconds_taken": round(time.perf_counter() - started, 1),
        }
    except Exception as exc:  # one bad file is reported, never fatal
        flac.unlink(missing_ok=True)
        meta_path.unlink(missing_ok=True)
        return {**base, "status": "failed", "error": f"{type(exc).__name__}: {exc}"}

In [ ]:
# The harness's own cutting code, byte for byte, at HARNESS_COMMIT: ingest's stage 1 and 2 and the
# two helpers they import. Empty package files stand in for the rest of the backend.
import hashlib
import urllib.request

HARNESS = FT / "harness"
for rel, digest in HARNESS_SHA256.items():
    data = urllib.request.urlopen("https://raw.githubusercontent.com/Sagyam/labeling-harness/"
                                  f"{HARNESS_COMMIT}/{rel}").read()
    assert hashlib.sha256(data).hexdigest() == digest, f"{rel}: sha256 mismatch"
    path = HARNESS / rel.removeprefix("backend/")
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_bytes(data)
for pkg in ("app", "app/services", "app/services/ingest", "app/utils"):
    (HARNESS / pkg / "__init__.py").touch()
sys.path.insert(0, str(HARNESS))
sys.path.insert(0, str(FT))

import distill
import ftkit
import prekit
from app.services.silero_vad import SileroVAD

assert SileroVAD().available, "the Silero model did not load"
print("harness code at", HARNESS_COMMIT[:7], "checked:", len(HARNESS_SHA256), "files")

## The recordings

In [ ]:
import zipfile

from huggingface_hub import HfApi, hf_hub_download

TOKEN = os.environ["HF_TOKEN"]
api = HfApi(token=TOKEN)
with ftkit.timed(f"downloading {ZIP_NAME}"):
    archive = hf_hub_download(DATASET_REPO, ZIP_NAME, repo_type="dataset", token=TOKEN)
MP3 = FT / "recordings"
with ftkit.timed(f"unzipping {Path(archive).stat().st_size / 2**30:.1f} GiB"):
    with zipfile.ZipFile(archive) as z:
        z.extractall(MP3)
suffixes = {".mp3", ".m4a", ".webm", ".opus", ".ogg", ".wav", ".flac", ".aac"}
files = sorted(p for p in MP3.rglob("*") if p.is_file() and p.suffix.lower() in suffixes
               and not p.name.startswith("._"))
ids = {}
for p in files:
    ids.setdefault(distill.source_from_filename(p.name)[0], []).append(p.name)
clashes = {k: v for k, v in ids.items() if len(v) > 1}
assert not clashes, f"two files would share a source id: {clashes}"
channels = sorted({distill.source_from_filename(p.name)[1] for p in files})
print(f"{len(files)} recordings, {sum(p.stat().st_size for p in files) / 2**30:.1f} GiB, "
      f"from {len(channels)} channels: {', '.join(channels)}")

## Cut, a chunk at a time

Every recording goes through `prekit.process_file`: ingest's `normalize_audio`, then its Silero VAD
and `segment_audio_to_slices`. Each chunk is uploaded as soon as it is cut. The line after each
chunk says how long the rest will take at the rate so far.

In [ ]:
import multiprocessing
import shutil
import time
from concurrent.futures import ProcessPoolExecutor

prefix = f"{PREFIX}/sources/"
done = {Path(f).stem for f in api.list_repo_files(DATASET_REPO, repo_type="dataset")
        if f.startswith(prefix) and f.endswith(".json")}
todo = [p for p in files if distill.source_from_filename(p.name)[0] not in done]
workers = WORKERS or os.cpu_count()
chunk = CHUNK or workers
total_bytes = sum(p.stat().st_size for p in todo)
print(f"{len(done)} recordings already cut, {len(todo)} to cut on {workers} processes, "
      f"{chunk} per uploaded chunk")
results, started, cut_bytes = [], time.perf_counter(), 0
with ProcessPoolExecutor(max_workers=workers, mp_context=multiprocessing.get_context("fork")) as pool:
    for number, k in enumerate(range(0, len(todo), chunk), 1):
        part = todo[k:k + chunk]
        stage = FT / "stage"
        shutil.rmtree(stage, ignore_errors=True)
        t0 = time.perf_counter()
        got = list(pool.map(prekit.process_file, [str(p) for p in part], [str(stage)] * len(part),
                            [BLOCKED_CHANNELS] * len(part)))
        results += got
        cut = [g for g in got if g["status"] == "prepared"]
        cut_bytes += sum(p.stat().st_size for p in part)
        rate = cut_bytes / (time.perf_counter() - started)
        print(f"chunk {number}: {len(cut)} cut, {sum(g['status'] == 'failed' for g in got)} failed, "
              f"{sum(g['status'] == 'blocked' for g in got)} blocked | "
              f"{sum(g['duration'] for g in cut) / 3600:.1f} h audio -> "
              f"{sum(g['speech_seconds'] for g in cut) / 3600:.1f} h speech, "
              f"{sum(g['clips'] for g in cut)} clips in {time.perf_counter() - t0:.0f} s | "
              f"about {(total_bytes - cut_bytes) / rate / 60:.0f} min left", flush=True)
        if cut:
            with ftkit.timed(f"chunk {number}: uploading {len(cut)} recordings"):
                api.upload_folder(repo_id=DATASET_REPO, repo_type="dataset", folder_path=str(stage),
                                  path_in_repo=PREFIX, commit_message=f"{PREFIX}: chunk {number}, {len(cut)} recordings")
for g in results:
    if g["status"] != "prepared":
        print(f"{g['status'].upper()} {g['file']}: {g.get('error', 'channel ' + g['channel'])}")

## The manifest and the summary

Rebuilt from every `distill/sources/*.json` on the repo, so it is right whatever earlier runs
did.

In [ ]:
names = sorted(f for f in api.list_repo_files(DATASET_REPO, repo_type="dataset")
               if f.startswith(f"{PREFIX}/sources/") and f.endswith(".json"))
with ftkit.timed(f"reading {len(names)} recordings' clip lists"):
    sources = [json.loads(Path(hf_hub_download(DATASET_REPO, n, repo_type="dataset", token=TOKEN)).read_text("utf-8"))
               for n in names]
rows = [r for s in sources for r in s["rows"]]
per_channel = {}
for s in sources:
    c = per_channel.setdefault(s["channel"], {"recordings": 0, "audio_h": 0.0, "speech_h": 0.0, "clips": 0})
    c["recordings"] += 1
    c["audio_h"] += s["duration"] / 3600
    c["speech_h"] += s["speech_seconds"] / 3600
    c["clips"] += s["clips"]
summary = {
    "recordings": len(sources),
    "clips": len(rows),
    "audio_hours": round(sum(s["duration"] for s in sources) / 3600, 2),
    "speech_hours": round(sum(r["duration"] for r in rows) / 3600, 2),
    "vad": sorted({s["vad"] for s in sources}),
    "harness_commit": HARNESS_COMMIT,
    "blocked_channels": BLOCKED_CHANNELS,
    "per_channel": {k: {x: round(y, 2) if isinstance(y, float) else y for x, y in v.items()}
                    for k, v in sorted(per_channel.items())},
}
out = FT / "manifest"
out.mkdir(exist_ok=True)
(out / "clips.jsonl").write_text("".join(json.dumps(r, ensure_ascii=False) + "\n" for r in rows), "utf-8")
(out / "summary.json").write_text(json.dumps(summary, indent=1, ensure_ascii=False), "utf-8")
with ftkit.timed("uploading clips.jsonl and summary.json"):
    api.upload_folder(repo_id=DATASET_REPO, repo_type="dataset", folder_path=str(out), path_in_repo=PREFIX,
                      commit_message=f"{PREFIX}: manifest, {len(rows)} clips, {summary['speech_hours']} h")
print(json.dumps({k: v for k, v in summary.items() if k != "per_channel"}, indent=1))
for name, c in summary["per_channel"].items():
    print(f"  {name:<40} {c['recordings']:>4} recordings {c['audio_h']:>7.1f} h audio "
          f"{c['speech_h']:>7.1f} h speech {c['clips']:>7} clips")